# Causal (Masked) Self-Attention

This notebook builds on [self-attention-with-trainable-weights](2-self-attention-with-trainable-weights.ipynb). There, every token was allowed to attend to *every* other token in the sentence, including tokens that come later. That's fine for understanding a sentence that already fully exists, but it breaks down the moment we want to *generate* text one word at a time.

**Causal attention** (also called *masked* attention) fixes this by only letting each token attend to itself and the tokens that came before it -- never the ones that come after. As before, we'll implement every new idea first with explicit `for` loops so you can see exactly what's happening to each number, and then rewrite it using fast, vectorized PyTorch operations, checking along the way that both give identical results.

## 1. Why hide future tokens at all?

Think about what a language model is actually trained to do: look at the words so far, and predict the *next* word. At generation time, this is the only thing that's physically possible -- when the model is writing word 6, words 7, 8, 9... don't exist yet. It has no choice but to make its decision using only what came before.

Now imagine training this model *without* restricting attention. During training, the full sentence is available all at once, so if we let token 2 "peek" at token 5 through attention, the model could just learn to copy the answer instead of learning to actually predict it. That would make training trivially easy and completely useless -- the model would fail the moment it had to generate real text one word at a time, since future words wouldn't exist yet to peek at.

So, to keep training consistent with how the model will actually be used, we deliberately hide future tokens during training too: when computing attention for token $i$, we only allow it to look at tokens $0, 1, \dots, i$ (itself included), never $i+1, i+2, \dots$

## 2. Setup: Picking Up Where We Left Off

We reuse the same 5-word sentence and the same kind of `nn.Linear`-based query/key/value projections from the previous notebook, so we already have a set of *unmasked* attention weights to start from. (Every step behind this recap -- projecting to queries/keys/values, scoring, scaling, softmax -- was already implemented and verified by hand in the previous notebook, so we won't repeat the loops for it here.)

In [1]:
import torch
import torch.nn as nn

torch.manual_seed(42)
sentence = "Hello my name is Ahtesham"
tokens = sentence.split()
num_tokens = len(tokens)
d_in = 3   # input embedding size
d_out = 2  # query/key/value size

token_embeddings = torch.rand(num_tokens, d_in)

torch.manual_seed(789)
W_query = nn.Linear(d_in, d_out, bias=False)
W_key   = nn.Linear(d_in, d_out, bias=False)
W_value = nn.Linear(d_in, d_out, bias=False)

queries = W_query(token_embeddings)
keys    = W_key(token_embeddings)
values  = W_value(token_embeddings)

d_k = keys.shape[-1]
attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores / d_k**0.5, dim=-1)

print("Tokens:", tokens)
print("\nUnmasked attention weights (each row sums to 1):")
print(attn_weights)
print("\nRow sums:", attn_weights.sum(dim=-1))

Tokens: ['Hello', 'my', 'name', 'is', 'Ahtesham']

Unmasked attention weights (each row sums to 1):
tensor([[0.1835, 0.2228, 0.2016, 0.1709, 0.2212],
        [0.1870, 0.2225, 0.1984, 0.1727, 0.2193],
        [0.1849, 0.2268, 0.1976, 0.1681, 0.2227],
        [0.1864, 0.2206, 0.2003, 0.1741, 0.2186],
        [0.1849, 0.2260, 0.1981, 0.1687, 0.2223]], grad_fn=<SoftmaxBackward0>)

Row sums: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000], grad_fn=<SumBackward1>)


Notice that every row already sums to 1 -- but every row also freely mixes in *every* column, including columns that represent tokens later in the sentence. Row 0 (`"Hello"`) is currently paying attention to `"my"`, `"name"`, `"is"`, and `"Ahtesham"` -- words that, at generation time, wouldn't exist yet when `"Hello"` is being processed. That's exactly what we need to fix.

## 3. Building a Mask That Keeps Only the Past and Present

We want a grid of 1s and 0s the same shape as our attention weights: a 1 wherever a key token is allowed (at or before the query's position), and a 0 wherever it should be hidden (strictly after the query's position). This is exactly a **lower-triangular matrix**.

We build it two ways: by hand with a nested loop that checks, for every (query position `i`, key position `j`) pair, whether `j <= i`; and with PyTorch's built-in `torch.tril` ("triangle, lower"), which does the same thing in one call.

In [2]:
mask_manual = torch.zeros(num_tokens, num_tokens)
for i in range(num_tokens):        # i = query position (the "current" token)
    for j in range(num_tokens):    # j = key position (a token being attended to)
        if j <= i:                 # only allow tokens at or before the current one
            mask_manual[i, j] = 1.0

mask_tril = torch.tril(torch.ones(num_tokens, num_tokens))

print("Manual mask:")
print(mask_manual)
print("\ntorch.tril mask:")
print(mask_tril)
print("\nDo they match?", torch.equal(mask_manual, mask_tril))

Manual mask:
tensor([[1., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0.],
        [1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1.]])

torch.tril mask:
tensor([[1., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0.],
        [1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1.]])

Do they match? True


## 4. Method 1 -- Mask the Weights, Then Renormalize

The simplest approach: take the attention weights we already computed, multiply them elementwise by the mask (this zeroes out every "future" position), and then fix up each row so it sums back to 1.

**Why do we need to renormalize?** Zeroing out some entries in a row means that row no longer sums to 1 -- it sums to whatever is left. Since attention weights are supposed to represent "what percentage of attention goes to each token," we need to rescale the surviving entries so they once again add up to 100%. We do this by dividing every entry in a row by that row's new (smaller) sum.

**Does any "future" information leak through anyway?** It might look suspicious that we first ran a full softmax over *all* tokens (including future ones) and only masked things out afterward -- as if the future scores briefly influenced the computation. It turns out they don't, and we can see exactly why with a bit of simple algebra. Softmax computes `weight[i,j] = exp(score[i,j]) / sum_k exp(score[i,k])`. After we zero out the future entries and divide every surviving entry by the row's new sum, the following happens for any allowed position `j` (where `j <= i`):

$$\text{new\_weight}_{i,j} = \frac{\exp(\text{score}_{i,j}) / Z}{\sum_{k \le i} \exp(\text{score}_{i,k}) / Z} = \frac{\exp(\text{score}_{i,j})}{\sum_{k \le i} \exp(\text{score}_{i,k})}$$

The original normalizing constant $Z$ (the sum over *all* tokens, future included) cancels out completely! What's left is *exactly* the softmax you'd get if you had never included the future scores in the first place. In other words, masking-then-renormalizing produces the identical result to pretending the future tokens were never there -- there is no leftover trace of the future scores in the final weights.

In [3]:
# --- for-loop version ---
masked_scores_manual = torch.zeros(num_tokens, num_tokens)
for i in range(num_tokens):
    for j in range(num_tokens):
        masked_scores_manual[i, j] = attn_weights[i, j] * mask_manual[i, j]

row_sums_manual = torch.zeros(num_tokens)
for i in range(num_tokens):
    total = 0.0
    for j in range(num_tokens):
        total += masked_scores_manual[i, j].item()
    row_sums_manual[i] = total

masked_weights_manual = torch.zeros(num_tokens, num_tokens)
for i in range(num_tokens):
    for j in range(num_tokens):
        masked_weights_manual[i, j] = masked_scores_manual[i, j] / row_sums_manual[i]

# --- vectorized version ---
masked_scores = attn_weights * mask_tril
row_sums = masked_scores.sum(dim=-1, keepdim=True)
masked_weights = masked_scores / row_sums

print("Do the manual and vectorized versions match?",
      torch.allclose(masked_weights_manual, masked_weights))

print("\nMasked + renormalized attention weights:")
print(masked_weights)
print("\nRow sums (should all be 1):", masked_weights.sum(dim=-1))

Do the manual and vectorized versions match? True

Masked + renormalized attention weights:
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4567, 0.5433, 0.0000, 0.0000, 0.0000],
        [0.3035, 0.3722, 0.3243, 0.0000, 0.0000],
        [0.2385, 0.2823, 0.2563, 0.2228, 0.0000],
        [0.1849, 0.2260, 0.1981, 0.1687, 0.2223]], grad_fn=<DivBackward0>)

Row sums (should all be 1): tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000], grad_fn=<SumBackward1>)


## 5. Method 2 -- Mask With $-\infty$ *Before* Softmax (the Efficient Way)

Method 1 works, but it takes two full passes: one softmax over everything, then a second manual renormalization step. There's a cleverer way that gets the correctly-masked, correctly-normalized weights out of a *single* softmax call.

The trick relies on a simple mathematical fact about `exp`: as a number goes to negative infinity, `exp` of that number goes to exactly 0.

In [4]:
print("exp(-infinity) =", torch.exp(torch.tensor(-float('inf'))).item())
print("exp(-1000)      =", torch.exp(torch.tensor(-1000.0)).item())
print("exp(0)          =", torch.exp(torch.tensor(0.0)).item())

exp(-infinity) = 0.0
exp(-1000)      = 0.0
exp(0)          = 1.0


So here's the idea: instead of masking the *output* of softmax and fixing it up afterward, we set the disallowed attention **scores** to $-\infty$ *before* running softmax. When softmax computes `exp(score)` for one of these positions, it gets exactly `0` -- contributing nothing to its own weight, and nothing to the sum in the denominator either. Every allowed position, meanwhile, is completely unaffected. The result is a properly masked *and* properly normalized set of weights, straight out of one softmax call -- no separate renormalization step required, and it's also more numerically stable than dividing by a small row sum by hand.

First, let's build the *opposite* mask we need here: `True` wherever a position should be blanked out (i.e., strictly in the future, `j > i`), again by hand and with PyTorch's built-in `torch.triu` ("triangle, upper").

In [5]:
future_mask_manual = torch.zeros(num_tokens, num_tokens, dtype=torch.bool)
for i in range(num_tokens):
    for j in range(num_tokens):
        if j > i:                     # strictly future positions get hidden
            future_mask_manual[i, j] = True

future_mask_triu = torch.triu(torch.ones(num_tokens, num_tokens), diagonal=1).bool()

print("Do the two future-token masks match?",
      torch.equal(future_mask_manual, future_mask_triu))
print("\nfuture mask (True = hidden):")
print(future_mask_triu)

Do the two future-token masks match? True

future mask (True = hidden):
tensor([[False,  True,  True,  True,  True],
        [False, False,  True,  True,  True],
        [False, False, False,  True,  True],
        [False, False, False, False,  True],
        [False, False, False, False, False]])


In [6]:
masked_scores_inf = attn_scores.masked_fill(future_mask_triu, -torch.inf)
print("Attention scores with future positions set to -infinity:")
print(masked_scores_inf)

attn_weights_causal = torch.softmax(masked_scores_inf / d_k**0.5, dim=-1)
print("\nCausal attention weights (one softmax pass):")
print(attn_weights_causal)

print("\nDoes this match Method 1's mask-then-renormalize result?",
      torch.allclose(attn_weights_causal, masked_weights, atol=1e-6))

Attention scores with future positions set to -infinity:
tensor([[ 0.1371,    -inf,    -inf,    -inf,    -inf],
        [ 0.0999,  0.3455,    -inf,    -inf,    -inf],
        [ 0.1144,  0.4029,  0.2083,    -inf,    -inf],
        [ 0.1104,  0.3488,  0.2122,  0.0139,    -inf],
        [ 0.1163,  0.4005,  0.2142, -0.0136,  0.3769]],
       grad_fn=<MaskedFillBackward0>)

Causal attention weights (one softmax pass):
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4567, 0.5433, 0.0000, 0.0000, 0.0000],
        [0.3035, 0.3722, 0.3243, 0.0000, 0.0000],
        [0.2385, 0.2823, 0.2563, 0.2228, 0.0000],
        [0.1849, 0.2260, 0.1981, 0.1687, 0.2223]], grad_fn=<SoftmaxBackward0>)

Does this match Method 1's mask-then-renormalize result? True


Both methods agree exactly, confirming they're two ways of computing the same thing. Method 2 (masking with $-\infty$ before softmax) is the one actually used in practice, because it's a single, efficient, numerically stable pass.

## 6. Dropout -- Randomly Forgetting Some Connections on Purpose

**What is dropout, and why would we ever want it?** During training, `dropout` randomly zeroes out some fraction of values (here, some of the attention weights) on every forward pass. This might sound counterproductive, but the goal is to stop the model from becoming overly dependent on any single connection between tokens. If a particular attention link is randomly missing some of the time, the model is forced to spread out what it learns across multiple paths instead of leaning too hard on just one -- which tends to make it generalize better to new text it hasn't seen before. **Dropout is only active during training** -- once training is done, it's switched off so the model uses every connection it learned.

**Why rescale the surviving values?** If we zero out, say, 50% of the attention weights and do nothing else, the total amount of "signal" flowing through this layer drops by half -- which would confuse the layers built on top of it. To compensate, PyTorch's dropout multiplies every *surviving* value by $\frac{1}{1 - p}$ (where $p$ is the drop probability). That way, the *expected* total signal stays about the same whether dropout is zeroing things out or not, so nothing downstream needs to change behavior when dropout is switched off at the end of training.

Let's see this in action on a simple matrix of all 1s, using a (deliberately extreme, for-illustration-only) 50% drop rate.

In [7]:
torch.manual_seed(123)
p = 0.5
dropout = nn.Dropout(p)
example = torch.ones(6, 6)

print("Before dropout:")
print(example)
print("\nAfter dropout (p=0.5):")
print(dropout(example))

Before dropout:
tensor([[1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.]])

After dropout (p=0.5):
tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])


Notice that roughly half the entries became `0`, and every surviving entry became `2.0` instead of `1.0` -- that's exactly `1 / (1 - 0.5) = 2`, the rescaling we just described.

We can reproduce this same idea by hand: draw a random "keep" decision per entry with probability `1 - p`, then divide the surviving entries by `1 - p`. The exact 0/1 pattern won't match `nn.Dropout` (PyTorch draws its randomness in its own internal order), but the *behavior* -- roughly `p` fraction dropped, survivors scaled by `1/(1-p)` -- is identical.

In [8]:
torch.manual_seed(0)
keep_prob = 1 - p
keep_mask = torch.rand(6, 6) < keep_prob   # True = keep this entry
manual_dropout_output = example * keep_mask.float() / keep_prob

print("Manually-implemented dropout:")
print(manual_dropout_output)

# The exact fraction dropped on a tiny 6x6 example is noisy (only 36 coin flips),
# so let's check the statistic on a much bigger tensor to see it converge to p.
big_keep_mask = torch.rand(200, 200) < keep_prob
fraction_dropped = (~big_keep_mask).float().mean().item()
print("\nFraction dropped on a 200x200 example: {:.3f} (expected close to {:.2f})".format(
    fraction_dropped, p))

Manually-implemented dropout:
tensor([[2., 0., 2., 2., 2., 0.],
        [2., 0., 2., 0., 2., 2.],
        [2., 2., 2., 0., 0., 0.],
        [2., 2., 0., 0., 2., 0.],
        [2., 0., 0., 2., 2., 2.],
        [2., 0., 2., 2., 2., 2.]])

Fraction dropped on a 200x200 example: 0.502 (expected close to 0.50)


## 7. Putting It All Together: a `CausalAttention` Module

We now have every ingredient: query/key/value projections, a causal mask applied with $-\infty$ before softmax, and dropout applied to the resulting weights. Let's package all of it into a single reusable `nn.Module`, and make sure it works on a **batch** of inputs (more than one sentence at a time), since that's how real training data is fed to a model.

Two implementation details worth calling out:

* **`register_buffer`** -- the causal mask isn't a *trainable* parameter (it never gets updated by gradient descent), but it does need to travel with the model: if we later move the model to a GPU with `.to(device)`, we want the mask to move there automatically too, instead of silently staying on the CPU and causing a device-mismatch error. `register_buffer` tells PyTorch "keep track of this tensor as part of the model's state, but don't treat it as a learnable parameter."
* **`masked_fill_`** (with a trailing underscore) -- performs the fill *in place*, modifying the existing tensor directly rather than allocating a new one. This is simply a small memory/performance optimization.

In [9]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        # A buffer: travels with the model (e.g. to a GPU), but is not a trainable parameter.
        self.register_buffer(
            'mask',
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape   # b = batch size (how many sentences at once)
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2)   # transpose only the token dims, keep the batch dim
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf
        )
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values
        return context_vec

To check that this works on more than a single sentence at a time, let's simulate a batch by simply stacking our one sentence with itself, giving a batch of 2 identical sentences.

In [10]:
batch = torch.stack((token_embeddings, token_embeddings), dim=0)
print("Batch shape:", batch.shape, "-> (batch_size, num_tokens, d_in)")

torch.manual_seed(123)
context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length, dropout=0.0)
context_vecs = ca(batch)

print("\ncontext_vecs.shape:", context_vecs.shape, "-> (batch_size, num_tokens, d_out)")
print("\nContext vectors:")
print(context_vecs)
print("\nBoth items in the batch match (since they're the same sentence twice)?",
      torch.allclose(context_vecs[0], context_vecs[1]))

Batch shape: torch.Size([2, 5, 3]) -> (batch_size, num_tokens, d_in)

context_vecs.shape: torch.Size([2, 5, 2]) -> (batch_size, num_tokens, d_out)

Context vectors:
tensor([[[-0.8340, -0.3584],
         [-0.7841, -0.2061],
         [-0.7248, -0.1467],
         [-0.6725, -0.1607],
         [-0.6983, -0.1409]],

        [[-0.8340, -0.3584],
         [-0.7841, -0.2061],
         [-0.7248, -0.1467],
         [-0.6725, -0.1607],
         [-0.6983, -0.1409]]], grad_fn=<UnsafeViewBackward0>)

Both items in the batch match (since they're the same sentence twice)? True


## Conclusion

This notebook extended plain self-attention into **causal self-attention**, the form actually used to train autoregressive language models. Here's the "why" behind every new idea, in one line each:

* **Hide future tokens** -- because at generation time future words don't exist yet, so training must respect the same limitation, or the model would learn to cheat instead of predict.
* **Lower-triangular mask** -- a simple grid of 1s and 0s that captures "which key positions are allowed for each query position" (only the current position and everything before it).
* **Mask, then renormalize (Method 1)** -- zeroing disallowed weights and rescaling the row back to sum to 1 turns out to be mathematically identical to never having considered the future tokens at all, so no information leaks through.
* **Mask with $-\infty$ before softmax (Method 2)** -- relies on `exp(-infinity) = 0` to get the same masked-and-normalized result in a single, more efficient softmax pass.
* **Dropout** -- randomly zeroes some attention weights during training (rescaling the rest by $1/(1-p)$) so the model can't over-rely on any one connection, which tends to help it generalize better.
* **`register_buffer`** -- lets the mask travel with the model (e.g. across devices) without treating it as something to be learned.

Together, these pieces form the `CausalAttention` module: the direct predecessor to the multi-head attention mechanism used inside GPT-style models.